# 02 — Xử lý dữ liệu

**Mục tiêu:** loại cột rò rỉ và dòng sai, tính số phòng trong cùng lượt đặt, chia dữ
liệu **theo thời gian** và lưu schema dữ liệu đầu vào.

| Tập | Ngày đến | Dùng cho |
|---|---|---|
| `train.csv` | 07/2015 – 12/2016 | Cross validation, tinh chỉnh, fit mô hình |
| `test.csv` | 01/2017 – 04/2017 | Đánh giá một lần duy nhất ở notebook 03 |
| `production_holdout.csv` | 05/2017 – 08/2017 | **Không dùng** trong huấn luyện và đánh giá. Giữ lại để mô phỏng dữ liệu tương lai mà mô hình chưa từng thấy |

Dữ liệu được lưu ở **dạng cột gốc**: việc tạo đặc trưng nằm bên trong pipeline của mô
hình (`hotel_booking.features.BookingFeatureBuilder`), nên mô hình đã lưu nhận trực tiếp
một bản ghi đặt phòng và luôn xử lý giống hệt lúc huấn luyện.

**Bước nào nằm ở đâu.** Notebook này chỉ làm các bước **không học thống kê từ dữ liệu**. Mọi
bước học thống kê nằm trong pipeline của mô hình và được fit lại trên phần train của từng fold,
nên test và fold validation không bao giờ góp vào các thống kê đó (best practice #3 trong tài
liệu *Machine Learning System*: *use statistics from the train split only*).

| Bước | Ở đâu | Học từ dữ liệu? |
|---|---|---|
| Loại cột rò rỉ, loại dòng bất thường | notebook 02 (`data.clean_bookings`) | Không: ngưỡng cố định |
| Số phòng trong lượt đặt `rooms_in_booking` | notebook 02 (`data.add_group_size`) | Không: đếm dòng giống nhau, không dùng nhãn |
| Chia theo thời gian | notebook 02 (`data.time_split`) | Không |
| Đặc trưng từng bản ghi: tổng số đêm, giá theo đầu người, chia khoảng `lead_time`, ghép đặc trưng... | pipeline: `BookingFeatureBuilder` | Không |
| Giá tương đối `adr_relative` | pipeline: `BookingFeatureBuilder` | Có: giá trung vị |
| Điền giá trị thiếu, log1p + chuẩn hóa | pipeline: `modeling.make_preprocessor` | Có: trung vị, trung bình, độ lệch chuẩn |
| One-hot (gộp nhóm hiếm), native categorical, target encoding | pipeline: `modeling.make_preprocessor` | Có: danh sách nhóm, tỷ lệ hủy từng nhóm |
| Trọng số theo thời gian / theo lượt đặt | pipeline: `modeling.RecencyWeightedClassifier` | Có: ngày đến mới nhất |

Schema dữ liệu đầu vào (mục 4) cũng chỉ học từ train.

> Logic dùng chung (làm sạch, tạo đặc trưng, chia dữ liệu, pipeline mô hình) nằm trong
> module `src/hotel_booking/` và được kiểm thử bằng `pytest` trong thư mục `tests/`.
> Notebook chỉ gọi lại các hàm đó, nhờ vậy quá trình huấn luyện và dự đoán luôn dùng
> chung một cách xử lý.

In [1]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from hotel_booking import config, data

RANDOM_STATE = config.RANDOM_STATE
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "grid.alpha": 0.4})
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Màu cố định theo thực thể: cùng một ý nghĩa luôn cùng một màu trong mọi notebook.
COLORS = {"primary": "#2a78d6", "not_canceled": "#2a78d6", "canceled": "#eb6834",
          "secondary": "#1baf7a", "reference": "#52514e"}

PROJECT_ROOT = config.find_project_root()
RAW_DATA_PATH = PROJECT_ROOT / "dataset" / "raw" / "hotel_bookings.csv"
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: .

In [2]:
from hotel_booking.features import BookingFeatureBuilder

df = data.load_raw(RAW_DATA_PATH)
raw_sha256 = data.file_sha256(RAW_DATA_PATH)
print(f"Shape: {df.shape}")
print(f"SHA-256 của file gốc: {raw_sha256}")

Shape: (119390, 32)


SHA-256 của file gốc: 7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06


## 1. Phân loại cột

Mỗi cột gốc thuộc đúng một nhóm. Danh sách được khai báo trong `hotel_booking.config`.

In [3]:
column_groups = {
    "Đầu vào mô hình (biết lúc đặt phòng)": config.INPUT_COLUMNS,
    "Sau thời điểm đặt (giữ để phân tích, không làm đầu vào)": config.POST_BOOKING_COLUMNS,
    "Rò rỉ trực tiếp (loại bỏ)": config.LEAKAGE_COLUMNS,
    "Biến đích": [config.TARGET],
}
display(pd.DataFrame([(group, len(columns), ", ".join(columns)) for group, columns in column_groups.items()],
                     columns=["nhóm", "số cột", "cột"]))

all_grouped = [column for columns in column_groups.values() for column in columns]
assert len(all_grouped) == len(set(all_grouped)), "Một cột thuộc nhiều nhóm"
assert set(all_grouped) == set(df.columns), f"Cột chưa phân loại: {set(df.columns) - set(all_grouped)}"
print("Mọi cột gốc đều được phân loại đúng một lần.")

,nhóm,số cột,cột
0,Đầu vào mô hình (biết lúc đặt phòng),25,"hotel, arrival_date_month, meal, market_segmen..."
1,"Sau thời điểm đặt (giữ để phân tích, không làm...",4,"assigned_room_type, booking_changes, required_..."
2,Rò rỉ trực tiếp (loại bỏ),2,"reservation_status, reservation_status_date"
3,Biến đích,1,is_canceled


Mọi cột gốc đều được phân loại đúng một lần.


## 2. Làm sạch và tính số phòng trong cùng lượt đặt

In [4]:
clean_df, cleaning_log = data.clean_bookings(df)
display(cleaning_log)
print(f"Còn lại {len(clean_df):,} / {len(df):,} dòng")
display(clean_df[config.GROUP_SIZE_COLUMN].describe().to_frame().T)

,bước,số dòng bị loại
0,Tổng số khách bằng 0,180
1,adr âm hoặc lớn hơn 1000,2
2,children hoặc babies lớn hơn 5,3


Còn lại 119,205 / 119,390 dòng


,count,mean,std,min,25%,50%,75%,max
rooms_in_booking,"119,205.0000",8.3693,18.6952,1.0000,1.0000,1.0000,4.0000,180.0000


In [5]:
assert not set(config.LEAKAGE_COLUMNS) & set(clean_df.columns)
assert clean_df[config.ID_COLUMN].is_unique
assert set(clean_df[config.TARGET].unique()) == {0, 1}
guests = clean_df["adults"] + clean_df["children"].fillna(0) + clean_df["babies"]
assert (guests > 0).all() and clean_df["adr"].between(0, 1000).all()
assert (clean_df[config.GROUP_SIZE_COLUMN] >= 1).all()
print("Tất cả kiểm tra đều đạt.")

Tất cả kiểm tra đều đạt.


## 3. Chia dữ liệu theo thời gian

In [6]:
splits = data.time_split(clean_df)
split_summary = pd.DataFrame({
    name: {
        "rows": len(frame),
        "from": f"{frame[config.TIME_COLUMN].min():%Y-%m-%d}",
        "to": f"{frame[config.TIME_COLUMN].max():%Y-%m-%d}",
        "cancel_rate": round(float(frame[config.TARGET].mean()), 4),
    }
    for name, frame in splits.items()
}).T
display(split_summary)

for name, frame in splits.items():
    path = PROCESSED_DIR / f"{name}.csv"
    frame.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"{name:<20} {frame.shape} -> {path}")

,rows,from,to,cancel_rate
train,78586,2015-07-01,2016-12-31,0.3622
test,18442,2017-01-01,2017-04-30,0.3653
production_holdout,22177,2017-05-01,2017-08-31,0.4057


train                (78586, 33) -> .\dataset\processed\train.csv
test                 (18442, 33) -> .\dataset\processed\test.csv
production_holdout   (22177, 33) -> .\dataset\processed\production_holdout.csv


In [7]:
train_df, test_df, holdout_df = splits["train"], splits["test"], splits["production_holdout"]
assert sum(len(frame) for frame in splits.values()) == len(clean_df)
assert train_df[config.TIME_COLUMN].max() < test_df[config.TIME_COLUMN].min()
assert test_df[config.TIME_COLUMN].max() < holdout_df[config.TIME_COLUMN].min()
for left, right in [("train", "test"), ("train", "production_holdout"), ("test", "production_holdout")]:
    assert not set(splits[left][config.ID_COLUMN]) & set(splits[right][config.ID_COLUMN])

# Dòng trùng nội dung không được nằm ở hai tập khác nhau.
def content_hashes(frame: pd.DataFrame) -> set:
    return set(pd.util.hash_pandas_object(frame.drop(columns=config.ID_COLUMN), index=False))

assert not content_hashes(train_df) & content_hashes(test_df)
assert not content_hashes(train_df) & content_hashes(holdout_df)

# File đọc lại phải giống dữ liệu vừa ghi.
reloaded = data.read_split(PROCESSED_DIR, "train")
assert reloaded.shape == train_df.shape and reloaded[config.ID_COLUMN].equals(train_df[config.ID_COLUMN])
print("Tất cả kiểm tra đều đạt.")

Tất cả kiểm tra đều đạt.


## 4. Schema dữ liệu đầu vào và kiểm tra dữ liệu mới

Schema (giá trị hợp lệ của cột phân loại, khoảng giá trị của cột số) được học từ train;
`arrival_date_year` (luôn tăng theo thời gian) và `agent`, `company` (mã định danh) chỉ được
kiểm tra kiểu dữ liệu.
Kiểm tra test và production holdout theo schema cho thấy dữ liệu mới có những điểm khác
train: đây là các tình huống pipeline phải xử lý được (nhóm chưa từng thấy được gộp vào
nhóm "hiếm", giá trị ngoài khoảng vẫn được dự đoán bình thường).

In [8]:
input_schema = data.build_input_schema(train_df)
for name in ["test", "production_holdout"]:
    issues = data.validate_inputs(splits[name], input_schema)
    print(f"{name}: {len(issues)} vấn đề")
    if not issues.empty:
        display(issues)
assert data.validate_inputs(train_df, input_schema).empty

test: 1 vấn đề


,column,issue,rows
0,previous_bookings_not_canceled,ngoài khoảng của train,3


production_holdout: 2 vấn đề


,column,issue,rows
0,previous_bookings_not_canceled,ngoài khoảng của train,11
1,adr,ngoài khoảng của train,1


## 5. Minh họa đặc trưng được tạo trong pipeline

`BookingFeatureBuilder` chuẩn hóa giá trị gốc (`Undefined` → `SC`, `agent` trống → `none`,
`children` trống → 0) và tạo đặc trưng như tổng số đêm, tổng số khách, giá theo đầu người, tỷ
lệ hủy trước đây, tháng và thứ của ngày đến. Notebook 03 thử thêm các đặc trưng tùy chọn rút ra
từ EDA: chia khoảng `lead_time` (`lead_time_bins`), ghép đặc trưng (`crosses`) và giá tương đối
(`relative_adr`, học giá trung vị từ dữ liệu fit).

In [9]:
builder = BookingFeatureBuilder().fit(train_df)
features = builder.transform(train_df[config.INPUT_COLUMNS + [config.GROUP_SIZE_COLUMN]])
print(f"{len(builder.numeric_features())} đặc trưng số: {builder.numeric_features()}")
print(f"{len(builder.categorical_features())} đặc trưng phân loại: {builder.categorical_features()}")
display(features.head())
assert features[builder.numeric_features()].isna().sum().sum() == 0

optional = BookingFeatureBuilder(lead_time_bins=True, crosses=True, relative_adr=True).fit(train_df)
optional_columns = [name for name in optional.feature_names_out_ if name not in builder.feature_names_out_]
print(f"Đặc trưng tùy chọn: {optional_columns}")
display(optional.transform(train_df.head())[["hotel", "deposit_type", "lead_time", "adr"] + optional_columns])

21 đặc trưng số: ['lead_time', 'arrival_date_week_number', 'stays_in_weekend_nights', 'stays_in_week_nights', 'total_nights', 'adults', 'children', 'babies', 'total_guests', 'is_family', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'previous_cancel_ratio', 'days_in_waiting_list', 'adr', 'adr_per_guest', 'total_of_special_requests', 'has_agent', 'has_company', 'rooms_in_booking']
10 đặc trưng phân loại: ['hotel', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type', 'agent', 'arrival_month', 'arrival_weekday']


,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,total_nights,adults,children,babies,total_guests,is_family,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,previous_cancel_ratio,days_in_waiting_list,adr,adr_per_guest,total_of_special_requests,has_agent,has_company,rooms_in_booking,hotel,meal,market_segment,distribution_channel,reserved_room_type,deposit_type,customer_type,agent,arrival_month,arrival_weekday
0,342,27,0,0,0,2,0.0000,0,2.0000,0,0,0,0,0.0000,0,0.0000,0.0000,0,0,0,1,Resort Hotel,BB,Direct,Direct,C,No Deposit,Transient,none,7,2
1,737,27,0,0,0,2,0.0000,0,2.0000,0,0,0,0,0.0000,0,0.0000,0.0000,0,0,0,1,Resort Hotel,BB,Direct,Direct,C,No Deposit,Transient,none,7,2
2,7,27,0,1,1,1,0.0000,0,1.0000,0,0,0,0,0.0000,0,75.0000,75.0000,0,0,0,1,Resort Hotel,BB,Direct,Direct,A,No Deposit,Transient,none,7,2
3,13,27,0,1,1,1,0.0000,0,1.0000,0,0,0,0,0.0000,0,75.0000,75.0000,0,1,0,1,Resort Hotel,BB,Corporate,Corporate,A,No Deposit,Transient,304,7,2
4,14,27,0,2,2,2,0.0000,0,2.0000,0,0,0,0,0.0000,0,98.0000,49.0000,1,1,0,2,Resort Hotel,BB,Online TA,TA/TO,A,No Deposit,Transient,240,7,2


Đặc trưng tùy chọn: ['adr_relative', 'lead_time_bucket', 'hotel_x_deposit_type', 'market_segment_x_deposit_type', 'hotel_x_market_segment']


,hotel,deposit_type,lead_time,adr,adr_relative,lead_time_bucket,hotel_x_deposit_type,market_segment_x_deposit_type,hotel_x_market_segment
0,Resort Hotel,No Deposit,342,0.0000,0.0000,181-365,Resort Hotel|No Deposit,Direct|No Deposit,Resort Hotel|Direct
1,Resort Hotel,No Deposit,737,0.0000,0.0000,>365,Resort Hotel|No Deposit,Direct|No Deposit,Resort Hotel|Direct
2,Resort Hotel,No Deposit,7,75.0000,0.6219,0-7,Resort Hotel|No Deposit,Direct|No Deposit,Resort Hotel|Direct
3,Resort Hotel,No Deposit,13,75.0000,0.6219,8-30,Resort Hotel|No Deposit,Corporate|No Deposit,Resort Hotel|Corporate
4,Resort Hotel,No Deposit,14,98.0000,0.8126,8-30,Resort Hotel|No Deposit,Online TA|No Deposit,Resort Hotel|Online TA


## 6. Lưu đặc tả dữ liệu

`feature_spec.json` là nguồn duy nhất cho danh sách cột đầu vào, schema và mốc chia;
notebook 03 và mọi chương trình dùng lại mô hình đọc từ file này.

In [10]:
feature_spec = {
    "dataset": "hotel_bookings.csv (Kaggle: jessemostipak/hotel-booking-demand)",
    "raw_sha256": raw_sha256,
    "target": config.TARGET,
    "id_column": config.ID_COLUMN,
    "time_column": config.TIME_COLUMN,
    "input_columns": config.INPUT_COLUMNS + [config.GROUP_SIZE_COLUMN],
    "post_booking_columns": config.POST_BOOKING_COLUMNS,
    "leakage_columns_removed": config.LEAKAGE_COLUMNS,
    "engineered_features": {"numeric": builder.numeric_features(), "categorical": builder.categorical_features()},
    "cleaning_log": cleaning_log.to_dict(orient="records"),
    "splits": {name: {"file": f"{name}.csv", **{k: (int(v) if k == "rows" else v) for k, v in row.items()}}
               for name, row in split_summary.iterrows()},
    "input_schema": input_schema,
}
spec_path = PROCESSED_DIR / "feature_spec.json"
spec_path.write_text(json.dumps(feature_spec, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print(f"Đã lưu {spec_path}")

Đã lưu .\dataset\processed\feature_spec.json

## Đầu ra

- `dataset/processed/train.csv`, `test.csv`: dữ liệu đã làm sạch ở dạng cột gốc, kèm
  `booking_id`, `arrival_date`, `rooms_in_booking`, các cột sau thời điểm đặt và biến đích.
- `dataset/processed/production_holdout.csv`: không được đọc trong huấn luyện và đánh giá.
- `dataset/processed/feature_spec.json`: cột đầu vào, schema, quy tắc làm sạch, mốc chia
  và mã băm của file gốc.